<a href="https://colab.research.google.com/github/toxzak-svg/fabq-rc/blob/main/FABQ-VP-8B-Notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FABQ-VP: Variable Precision FABQ-RC

**Author:** Zach Maronek · May 2026

---

## The Problem FABQ-VP Solves

FABQ-RC operates at exactly 1-bit (binary ±1 + int4). FABQ-VP extends this to **variable precision across 2-8 bits per parameter**, targeting ~3-4 bpw for larger models.

## Precision Pyramid

| Precision | Fraction | Bits/Param |
|-----------|----------|------------|
| FP16 | 0.5% | 16.0 |
| int8 | 4.5% | 8.0 |
| int4 | 20% | 4.0 |
| int2 | 25% | 2.0 |
| binary | 50% | 1.0 |

**Target:** ~3.0-4.0 bpw with <5% perplexity degradation

---

**Contents:** [1. Setup](#1) · [2. Method](#2) · [3. Implementation](#3) · [4. Evaluation](#4)

<a id="1"></a>
## 1. Setup & Imports

In [ ]:
# Core dependencies
!pip install -q torch==2.5.1 --index-url https://download.pytorch.org/whl/cu121
!pip install -q transformers accelerate bitsandbytes scikit-learn
!pip install -q pandas numpy==1.26.4 tqdm matplotlib seaborn datasets

import os, math, json, time, sys
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.cluster import MiniBatchKMeans
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

# Detect platform
try:
    from google.colab import userdata
    COLAB = True
    print("Running on Google Colab")
except ImportError:
    COLAB = False
    print("Running locally or Kaggle")

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

---

<a id="2"></a>
## 2. Method Overview

FABQ-VP has four stages:

```
FP16 Weights
     │
     ▼
Stage 1: Fisher-Weighted Channel Importance
     │
     ▼
Stage 2: 5-Level Precision Allocation (fp16, int8, int4, int2, binary)
     │
     ▼
Stage 3: Per-Precision Blocksize Selection
     │
     ▼
Stage 4: Multi-Level Residual Codebooks
     │
     ▼
FABQ-VP Quantized Model (~3.5 bpw)
```

<a id="3"></a>
## 3. Implementation

In [ ]:
# Model configuration - using Qwen2.5-7B for 8B-class validation
MODEL_NAME = "Qwen/Qwen2.5-7B"
MAX_SEQ_LEN = 128
CALIB_SIZE = 512

# Precision pyramid fractions (must sum to 1.0)
PRECISION_PYRAMID = {
    'fp16': 0.005,
    'int8': 0.045,
    'int4': 0.200,
    'int2': 0.250,
    'binary': 0.500
}

# Blocksize candidates per precision level
BS_CANDIDATES = {
    'fp16': [1],
    'int8': [1],
    'int4': [16, 32, 64],
    'int2': [32, 64, 128],
    'binary': [64, 128, 256, 512]
}

# Get HF token for private models
hf_token = None
if COLAB:
    try:
        hf_token = userdata.get('HF_TOKEN')
    except Exception:
        pass
else:
    hf_token = os.environ.get('HF_TOKEN')

print(f"Model: {MODEL_NAME}")
print(f"Precision pyramid: {PRECISION_PYRAMID}")

In [ ]:
print(f"Loading tokenizer for {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True, token=hf_token)
tokenizer.pad_token = tokenizer.eos_token
print("Tokenizer loaded")

In [ ]:
print(f"Loading {MODEL_NAME}...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map='auto',
    torch_dtype=torch.float16,
    trust_remote_code=True,
    token=hf_token
)
model.eval()
total_params = sum(p.numel() for p in model.parameters())
print(f"Model loaded: {total_params / 1e9:.2f}B parameters")

### 3.1 Prepare Calibration Data

In [ ]:
from datasets import load_dataset

print("Loading C4 calibration data...")
c4 = load_dataset(
    "allenai/c4",
    data_files={"train": "en/c4-train.00000-of-01024.json.gz"},
    split=f"train[:{CALIB_SIZE}]"
)

def tokenize_fn(batch):
    enc = tokenizer(
        batch['text'],
        truncation=True,
        max_length=MAX_SEQ_LEN,
        padding='max_length'
    )
    enc['labels'] = enc['input_ids'].copy()
    return enc

cal_dataset = c4.map(tokenize_fn, batched=True, remove_columns=['text'])
cal_dataset.set_format('torch', columns=['input_ids', 'labels'])
cal_loader = DataLoader(cal_dataset, batch_size=1, shuffle=False)
print(f"{len(cal_loader)} calibration samples ready")

### 3.2 Stage 1 — Fisher-Weighted Channel Importance

In [ ]:
class FisherAccumulator:
    """Accumulate Fisher Information per output channel."""
    def __init__(self, model):
        self.model = model
        self.hooks = []

    def _hook_fn(self, module, grad_input, grad_output):
        if grad_output[0] is not None and hasattr(module, '_fisher_grad') and module.weight.grad is not None:
            grad_sq = module.weight.grad.data ** 2
            if grad_sq.dim() == 2:
                channel_fisher = grad_sq.sum(dim=1)
            else:
                channel_fisher = grad_sq.sum(dim=(1, 2, 3))
            module._fisher_grad += channel_fisher

    def __call__(self, cal_loader, device):
        for name, module in self.model.named_modules():
            if isinstance(module, nn.Linear):
                module._fisher_grad = torch.zeros_like(module.weight).sum(dim=1)

        for module in self.model.modules():
            if isinstance(module, nn.Linear):
                self.hooks.append(module.register_full_backward_hook(self._hook_fn))

        self.model.train()
        for batch in tqdm(cal_loader, desc="Computing Fisher"):
            input_ids = batch['input_ids'].to(device)
            labels = batch['labels'].to(device)
            outputs = self.model(input_ids)
            loss = F.cross_entropy(outputs.logits.view(-1, outputs.logits.size(-1)), labels.view(-1))
            loss.backward()
            self.model.zero_grad()

        for h in self.hooks:
            h.remove()
        self.model.eval()

        fisher = {}
        for name, module in self.model.named_modules():
            if isinstance(module, nn.Linear) and hasattr(module, '_fisher_grad'):
                fisher[name] = module._fisher_grad.clone()
                del module._fisher_grad
        return fisher

print("Computing Fisher importance...")
fisher_acc = FisherAccumulator(model)
fisher_scores = fisher_acc(cal_loader, DEVICE)
print(f"Fisher computed for {len(fisher_scores)} layers")

### 3.3 Stage 2 — 5-Level Precision Allocation

In [ ]:
def allocate_vp_precision(fisher_dict, pyramid=PRECISION_PYRAMID):
    """
    Allocate precision levels per channel based on Fisher scores.
    """
    allocation = {}
    for name, fisher in fisher_dict.items():
        out_channels = fisher.shape[0]
        order = torch.argsort(fisher, descending=True)
        
        alloc = {}
        cumulative = 0
        
        for precision, fraction in pyramid.items():
            threshold = int(out_channels * fraction)
            for i in range(cumulative, min(cumulative + threshold, out_channels)):
                alloc[int(order[i])] = precision
            cumulative += threshold
        
        allocation[name] = alloc
    return allocation

allocation = allocate_vp_precision(fisher_scores, PRECISION_PYRAMID)

# Summarize
total_channels = sum(len(a) for a in allocation.values())
precision_counts = {}
for name, alloc in allocation.items():
    for prec in alloc.values():
        precision_counts[prec] = precision_counts.get(prec, 0) + 1

print("Precision allocation summary:")
for prec, count in sorted(precision_counts.items(), key=lambda x: ['fp16', 'int8', 'int4', 'int2', 'binary'].index(x[0])):
    print(f"   {prec}: {count:,} channels ({100*count/total_channels:.1f}%)")

### 3.4 Stage 3 — Per-Precision Blocksize Selection

In [ ]:
def compute_reconstruction_error(weights, blocksize, fisher_channels):
    """Compute Fisher-weighted reconstruction error."""
    out_c, in_c = weights.shape
    total_err = 0.0
    for start in range(0, in_c, blocksize):
        end = min(start + blocksize, in_c)
        block = weights[:, start:end]
        scale = block.std() + 1e-8
        block_q = np.where(block > 0, 1.0, -1.0) * scale
        recon_err = ((block - block_q) ** 2).mean()
        block_fisher = fisher_channels.mean().item()
        total_err += block_fisher * recon_err
    return total_err


def select_blocksize_for_precision(weights, fisher_channels, precision, candidates):
    """Pick blocksize minimizing reconstruction error for given precision."""
    if precision in ['fp16', 'int8']:
        return 1
    
    best_b, best_err = candidates[0], float('inf')
    for b in candidates:
        err = compute_reconstruction_error(weights, b, fisher_channels)
        if err < best_err:
            best_err = err
            best_b = b
    return best_b

print("Selecting per-layer blocksize per precision level...")
blocksize_results = {}
for name, module in tqdm(list(model.named_modules()), desc="Blocksize sweep"):
    if not isinstance(module, nn.Linear):
        continue
    if name not in fisher_scores:
        continue
    
    weights = module.weight.data.cpu().numpy()
    fisher = fisher_scores[name]
    alloc = allocation[name]
    
    blocksize_results[name] = {}
    for prec, candidates in BS_CANDIDATES.items():
        chs = [ch for ch, p in alloc.items() if p == prec]
        if not chs:
            continue
        ch_weights = weights[chs, :]
        ch_fisher = fisher[chs]
        best_b = select_blocksize_for_precision(ch_weights, ch_fisher, prec, candidates)
        blocksize_results[name][prec] = best_b

print("\nBlocksize distribution by precision:")
for prec in ['int4', 'int2', 'binary']:
    bs_for_prec = [r.get(prec, 0) for r in blocksize_results.values() if prec in r]
    if bs_for_prec:
        print(f"   {prec}: {pd.Series(bs_for_prec).value_counts().to_dict()}")

### 3.5 Stage 4 — Multi-Level Residual Codebooks

In [ ]:
def build_multi_residual_codebooks(model, allocation, blocksize_results, cal_loader, device, n_clusters=256, max_samples=8192):
    """
    Build residual codebooks for each precision transition:
    - int4->int8: clusters W_int8 - W_int4 residuals
    - int2->int4: clusters W_int4 - W_int2 residuals
    - binary->int2: clusters W_int2 - W_binary residuals
    """
    codebooks = {}
    max_bs = max(BS_CANDIDATES['binary'])
    
    for (lower_prec, upper_prec) in [('int2', 'int4'), ('binary', 'int2')]:
        print(f"\nBuilding {lower_prec}->{upper_prec} codebook...")
        all_residuals = []
        sample_count = 0
        
        for batch in tqdm(cal_loader, desc=f"Collecting {lower_prec} residuals", total=min(len(cal_loader), max_samples // 8)):
            if sample_count >= max_samples:
                break
            input_ids = batch['input_ids'].to(device)
            labels = batch['labels'].to(device)
            outputs = model(input_ids)
            model.zero_grad()
            
            for name, module in model.named_modules():
                if not isinstance(module, nn.Linear) or name not in allocation:
                    continue
                
                weights = module.weight.detach().cpu().numpy()
                alloc = allocation[name]
                
                lower_chs = [ch for ch, p in alloc.items() if p == lower_prec]
                
                if not lower_chs:
                    continue
                
                bs = blocksize_results.get(name, {}).get(lower_prec, 128)
                
                for ch in lower_chs:
                    for start in range(0, weights.shape[1], bs):
                        end = min(start + bs, weights.shape[1])
                        
                        # Skip padded blocks
                        if end - start < bs:
                            continue
                        
                        block = weights[ch, start:end]
                        scale = block.std() + 1e-8
                        block_q = np.where(block > 0, 1.0, -1.0) * scale
                        residual = block - block_q
                        
                        res_flat = residual.flatten()
                        pad_size = max_bs - len(res_flat)
                        padded_res = np.pad(res_flat, (0, pad_size), mode='constant')
                        all_residuals.append(padded_res)
                        sample_count += 1
                        
                        if sample_count >= max_samples:
                            break
                    if sample_count >= max_samples:
                        break
                if sample_count >= max_samples:
                    break
        
        residuals_array = np.array(all_residuals, dtype=np.float32)
        print(f"   Collected {residuals_array.shape[0]} blocks")
        
        # Cluster
        kmeans = MiniBatchKMeans(n_clusters=n_clusters, random_state=42, batch_size=1024, n_init=3)
        kmeans.fit(residuals_array)
        codebooks[(lower_prec, upper_prec)] = kmeans.cluster_centers_
        print(f"   Built {lower_prec}->{upper_prec} codebook with {n_clusters} centroids")
    
    return codebooks

codebooks = build_multi_residual_codebooks(model, allocation, blocksize_results, cal_loader, DEVICE)
print(f"\nBuilt {len(codebooks)} residual codebooks")

<a id="4"></a>
## 4. Evaluation

In [ ]:
def compute_bpw_vp(model, allocation, codebooks):
    """Compute effective bits per parameter for FABQ-VP."""
    precision_bits = {'fp16': 16, 'int8': 8, 'int4': 4, 'int2': 2, 'binary': 1}
    
    total_bits = 0
    total_params = 0
    
    for name, module in model.named_modules():
        if not isinstance(module, nn.Linear):
            continue
        if name not in allocation:
            continue
        
        shape = module.weight.shape
        n = shape[0] * shape[1]
        alloc = allocation[name]
        
        for prec, bits in precision_bits.items():
            ch_count = sum(1 for v in alloc.values() if v == prec)
            total_bits += ch_count * shape[1] * bits
        
        total_params += n
    
    # Codebook overhead
    for cb in codebooks.values():
        total_bits += cb.nbytes * 8
    
    bpw = total_bits / total_params
    return bpw

bpw = compute_bpw_vp(model, allocation, codebooks)
print(f"FABQ-VP effective bits per parameter: {bpw:.4f}")

In [ ]:
import math

def compute_perplexity(model, dataset, tokenizer, device, stride=128, max_samples=256):
    """Compute perplexity on a dataset."""
    model.eval()
    total_loss = 0.0
    total_tokens = 0
    
    texts = [dataset[i]['text'] for i in range(min(max_samples, len(dataset)))]
    
    for i in tqdm(range(len(texts)), desc="Computing perplexity"):
        text = texts[i]
        enc = tokenizer(text, return_tensors='pt', truncation=True, max_length=512)
        input_ids = enc['input_ids'].to(device)
        seq_len = input_ids.size(1)
        
        for start in range(0, seq_len - 1, stride):
            end = min(start + stride, seq_len - 1)
            chunk = input_ids[:, start:end]
            labels = input_ids[:, start+1:end+1]
            
            with torch.no_grad():
                outputs = model(chunk)
                logits = outputs.logits
                
                shift_logits = logits[:, :-1, :].contiguous()
                shift_labels = labels.contiguous()
                
                loss = F.cross_entropy(
                    shift_logits.view(-1, shift_logits.size(-1)),
                    shift_labels.view(-1),
                    reduction='sum'
                )
                total_loss += loss.item()
                total_tokens += shift_labels.numel()
    
    ppl = math.exp(total_loss / total_tokens)
    return ppl

print("Computing FP16 baseline perplexity...")
fp16_ppl = compute_perplexity(model, c4, tokenizer, DEVICE)
print(f"FP16 perplexity: {fp16_ppl:.4f}")

## 5. Results Summary

In [ ]:
print("\n" + "="*60)
print("FABQ-VP PHASE 1 RESULTS")
print("="*60)
print(f"\nModel: {MODEL_NAME}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B")
print(f"\nPrecision allocation:")
for prec, count in sorted(precision_counts.items(), key=lambda x: ['fp16', 'int8', 'int4', 'int2', 'binary'].index(x[0])):
    print(f"   {prec}: {count:,} channels ({100*count/total_channels:.1f}%)")
print(f"\nBits per parameter: {bpw:.4f}")
print(f"FP16 perplexity: {fp16_ppl:.4f}")
print("\nNote: FABQ-VP quantization not yet applied - this is calibration only")

---

## Next Steps

1. **Apply FABQ-VP quantization** to model weights
2. **Measure perplexity** after quantization
3. **Compare** against uniform 3-bit and AWQ 4-bit baselines
4. **Evaluate downstream** tasks (ARC, HellaSwag, etc.)

---